## Requirements
- Serverless compute v4
- databricks_airbnb_sample_data: available from the Marketplace, downloaded as a catalog

## Setup

In [0]:
%pip install unitycatalog-ai[databricks]==0.3.2 -qqq
%restart_python

## A. Data and Function Client
The Databricks Function Client is a specialized interface for creating, managing and running UC functions for both SQL and Python.

In [0]:
# Define your parameters
catalog = "workspace"
schema = "bronze"
table_name = f"{catalog}.{schema}.sf_airbnb_listings"
avg_function_name = f"{catalog}.{schema}.avg_neigh_price"
cnt_function_name = f"{catalog}.{schema}.cnt_by_room_type"

In [0]:
df = spark.read.table(table_name)
display(df.limit(5))

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient


client = DatabricksFunctionClient(execution_mode="serverless")

## B. SQL Functions

### B1. Testing the SQL Logic

In [0]:
# test the logic that finds the average price of all listings in the Western Addition neighborhood:
display(spark.sql(f"""
    SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE)) AS avg_price
    FROM {table_name}
    WHERE neighbourhood_cleansed = 'Western Addition'
        AND price IS NOT NULL
        AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''
"""))

In [0]:
display(spark.sql(f"""
    SELECT COUNT(*)
    FROM {table_name}
    WHERE neighbourhood_cleansed = 'Western Addition'
    AND room_type = 'Private room'
"""))

### B2. Create or Update the Function from the Logic

In [0]:
# # Drop any existing functions with the same name
# spark.sql(f"""
# DROP FUNCTION IF EXISTS {avg_function_name};
# -- DROP FUNCTION IF EXISTS airbnb_posting_info;
# """)

In [0]:

# Create the function using spark.sql with f-strings
# Check if the function already exists and inform the user
# func_exists = spark.sql(f"""
#     SELECT COUNT(*) AS cnt
#     FROM information_schema.routines
#     WHERE routine_catalog = '{catalog}'
#       AND routine_schema = '{schema}'
#       AND routine_name = '{avg_function_name.split('.')[-1]}'
# """).collect()[0]['cnt'] > 0

def function_exists(function_name):
    result = spark.sql(f"SHOW USER FUNCTIONS IN {catalog}.{schema} LIKE '{function_name.split('.')[-1]}'")
    return result.count() > 0

if function_exists(avg_function_name):
    print(f"Function {avg_function_name} already exists and will be replaced.")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {avg_function_name}(
    neigh_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')"
)
RETURNS DOUBLE
DETERMINISTIC
COMMENT 'Calculate the average listing price for a specific neighborhood in San Francisco. Returns the average price as a numeric value. Price strings are cleaned and converted to numeric values before averaging.'
RETURN
SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE)) AS avg_price
FROM {table_name}
WHERE neighbourhood_cleansed = neigh_name
    AND price IS NOT NULL
    AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''
""")

print(f"Function {avg_function_name} has been created/updated successfully!")

In [0]:
if function_exists(cnt_function_name):
    print(f"Function {cnt_function_name} already exists and will be replaced.")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {cnt_function_name}(
    neigh_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')",
    room_type_filter STRING COMMENT "The room type to count (e.g., 'Private room' or 'Shared room')"
)
RETURNS BIGINT
DETERMINISTIC
COMMENT 'Counts the number of AirBnb listings for a specific room type in a given neighborhood. Returns the count as an integer.'
RETURN
SELECT COUNT(*)
    FROM {table_name}
    WHERE neighbourhood_cleansed = neigh_name
    -- need to use a different param name than the column name "room_type" to avoid SQL conflict
    -- when conflict, the WHERE clause is comparing the column to itself rather than to the parameter value, which will always be true for non-null rows.
    AND room_type = room_type_filter 
""")

print(f"Function {cnt_function_name} has been created/updated successfully!")

### B3. Test the SQL Tool Using SQL Syntax

In [0]:
%sql
SELECT workspace.bronze.avg_neigh_price('Mission') as mission_avg_price

In [0]:
%sql
SELECT workspace.bronze.cnt_by_room_type('Western Addition', 'Private room') as western_addition_avg_price

## C. Python Agent Tools

### C1. Build and Test Python Logics

In [0]:
def airbnb_posting_info(id: int) -> str:
    """
    Fetches AirBnb posting information as formatted text.

    Args:
      id (int): The ID of the AirBnb listing to fetch information for. E.g., (958)
    
    Returns:
      str: The formatted text containing the AirBnb posting information (description, reviews, rating) or error message.

    """
    import requests
    import re

    api_url = f"https://www.airbnb.com/rooms/{id}"

    try:
        response = requests.get(api_url, timeout=10)

        if response.status_code == 200:
            html = response.text

            # Extract description:
            desc = re.search(r'"metaDescription":"([^"]+)"', html)
            if desc:
                description = desc.group(1).replace('\\n', ' ')
                parts = description.split(' · ') # split by the middle dot char
                description = ' · '.join(parts[2:]) if len(parts) > 2 else description
            else:
                description = "Description not found" # Fallback if desc not found
            
            # Extract review count and rating using regex:
            reviews = re.search(r'"reviewCount":(\d+)', html)
            rating = re.search(r'"starRating":([\d.]+)', html)

            reviews = reviews.group(1) if reviews else "N/A"
            rating = rating.group(1) if rating else "N/A"

            return f"""Description: {description}
        Reviews: {reviews}
        Rating: {rating}
        """
        else:
            return f"Request failed with status code: {response.status_code}"
    except requests.exceptions.RequestException as e:
        return f"Request error: {str(e)}"


In [0]:
info = airbnb_posting_info(958)
print(info)

### C2. Register Python Function Tool using `DatabricksFunctionClient()`

Use `client.create_python_function()` with parameters:
- `func`: The Python function obj
- `catalog`: the catalog name
- `schema`: the schema name
- `replace`: set to True to overwrite if exists

In [0]:
function_info = client.create_python_function(
    func = airbnb_posting_info,
    catalog = catalog,
    schema = schema,
    replace = True
)

### C3. Test the Python Function with the Client

In [0]:
result = client.execute_function(
    function_name=f"{catalog}.{schema}.airbnb_posting_info",
    parameters={
        "id": 958
    }
)
print(result.value)

## D. Test the Functions as Agent Tool in Playground
- Open Playground
- Choose an LLM with tool-use ability
- Under "Tools", select UC functions, browse and select the registered functions above.
- Start chatting with it, e.g., 
    - Compare average prices between Haight Ashbury and Mission
    - Find listing info of ID 958